In [5]:
import pandas as pd
import sqlite3

pd.set_option('display.max_columns', None)

## Combine datasets

In [12]:
gaming_data = pd.read_csv('../data/gaming_GAD_CLEAN.csv')
nhis_data = pd.read_csv('../data/nhis_GAD_CLEAN.csv')

In [13]:
set(gaming_data.columns) ^ set(nhis_data.columns)

{'Degree',
 'Gender',
 'game',
 'hours',
 'participant_id_OLD',
 'platform',
 'playstyle',
 'reason',
 'region_id'}

In [22]:
df = pd.concat([gaming_data, nhis_data], ignore_index=True, sort=False)
df.head()

# Reference: https://pandas.pydata.org/docs/user_guide/merging.html

,Unnamed: 0,participant_id,year,GAD1,GAD2,GAD3,GAD4,GAD5,GAD6,GAD7,GAD_total,GAD_cat,sex_id,age,education_id,survey_id,residence,game,platform,hours,reason,playstyle,Gender,Degree,region_id,participant_id_OLD
0,0,1,2015,0,0,0,0,1,0,0,1,1,1,25,8,1,USA,Skyrim,"Console (PS, Xbox, ...)",15.0,having fun,Singleplayer,Male,Bachelor (or equivalent),NaN,NaN
1,1,2,2015,1,2,2,2,0,1,0,8,2,1,41,8,1,USA,Other,PC,8.0,having fun,Multiplayer - online - with strangers,Male,Bachelor (or equivalent),NaN,NaN
2,2,3,2015,0,2,2,0,0,3,1,8,2,2,32,8,1,DEU,Other,PC,0.0,having fun,Singleplayer,Female,Bachelor (or equivalent),NaN,NaN
3,3,4,2015,0,0,0,0,0,0,0,0,1,1,28,8,1,USA,Other,PC,20.0,improving,Multiplayer - online - with online acquaintanc...,Male,Bachelor (or equivalent),NaN,NaN
4,4,5,2015,2,1,2,2,2,3,2,14,3,1,19,12,1,KOR,Other,"Console (PS, Xbox, ...)",20.0,having fun,Multiplayer - online - with strangers,Male,High school diploma (or equivalent),NaN,NaN


## Build Relational Tables

In [ ]:
# ================
# Fact tables
# ================

participant_df = df[['participant_id', 'survey_id', 'sex_id', 'education_id', 'residence', 'age', 'region_id']].drop_duplicates()

GAD_response_df = df[['participant_id', 'GAD1', 'GAD2', 'GAD3', 'GAD4', 'GAD5', 'GAD6', 'GAD7', 'GAD_total', 'GAD_cat']]

gamers_df = df[['participant_id', 'survey_id', 'game', 'platform', 'hours', 'reason', 'playstyle']].drop_duplicates()

# ================
# Lookup tables
# ================

survey_df = pd.DataFrame({
    'survey_id': [1, 2],
    'survey_name': ['Gamers', 'NHIS'],
    'year': [2015, 2019],
    'population':['global', 'America']
})


GAD_categories = {
    1:'None/Minimal', 
    2:'Mild', 
    3:'Moderate', 
    4:'Severe', 
    8:'Not Ascertained'
}

GAD_cat_df = (
    pd.Series(GAD_categories, name='cat_name')
        .rename_axis('GAD_cat')
        .reset_index()
)

sex_lookup = {
    1: 'Male', 
    2:'Female', 
    7:'Refused', 
    8:'Not Ascertained',
    9:"Don't Know"
}
sex_df = (
    pd.Series(sex_lookup, name='sex_name')
        .rename_axis('sex_id')
        .reset_index()
)


degree_lookup = {
    0:'Never attended/kindergarten only', 
    1: 'Grade 1-11',
    2: '12th grade, no diploma',
    3: 'GED or equivalent',
    4: 'High School Graduate',
    5: 'Some college, no degree',
    6: 'Associate degree: occupational, technical, or vocational program',
    7: 'Associate degree: academic program',
    8: "Bachelor's degree (Example: BA, AB, BS, BBA)",
    9: "Master's degree or equivalent (Example: MA, MS, MEng, MEd, MBA)",
    10: "Professional School degree (Example: MD, DDS, DVM, JD)",
    11: "Doctoral degree (Example: PhD, EdD)",
    12: "High school diploma (or equivalent)",
    13: "Ph.D., Psy. D., MD (or equivalent)",
    97: "Refused",
    98: "Not Ascertained",
    99:"Don't Know"
}
education_df = (
    pd.Series(degree_lookup, name='level')
        .rename_axis('education_id')
        .reset_index()
)


region_lookup = {
    1: 'Northeast',
    2: 'Midwest',
    3: 'South', 
    4: 'West'
}
region_df = (
    pd.Series(region_lookup, name='name')
        .rename_axis('region_id')
        .reset_index()
)

# I conferred with ChatGPT to decide how to set up the lookup tables
# it recommended using dictionaries to set them up to make it easier to 
# read and maintain. I did this for the longer ones

In [ ]:
conn = sqlite3.connect('../data/mentalhealth.db')

participant_df.to_sql('participants', conn, index=False, if_exists='replace')
GAD_response_df.to_sql('responses', conn, index=False, if_exists='replace')
gamers_df.to_sql('gamers', conn, index=False, if_exists='replace')
survey_df.to_sql('survey', conn, index=False, if_exists='replace')
GAD_cat_df.to_sql('categories', conn, index=False, if_exists='replace')
sex_df.to_sql('sex', conn, index=False, if_exists='replace')
education_df.to_sql('education', conn, index=False, if_exists='replace')
region_df.to_sql('region', conn, index=False, if_exists='replace')

4